In [ ]:
import gc
from symbol import yield_arg

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns
import sys
sys.path.append("/extdata4/baeklab/Hyeonseo/m6A/modformer")
from utils.utils import printmessage
import glob
import tqdm
import gc

In [ ]:
plt.style.use('default')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 22, 'legend.facecolor': 'white', 'legend.framealpha': 1, "legend.frameon": 1, "lines.linewidth": 2})

In [ ]:
gff_path = "/extdata4/baeklab/Hyeonseo/m6A/anno/GCF_000001405.40_GRCh38.p14_genomic.gtf"

In [ ]:
columns = ["chr", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]

gff_path = pd.read_csv(refseq_path, sep= "\t", skiprows=5, names = columns, dtype={"chr":str})

print(gff_path)

In [ ]:
def ncid_to_chr(ncid):
    ncid_int=ncid.split(".")[0][3:]
    if not ncid_int.isnumeric():
        return "chrUnk"
    else:
        ncid_int = int(ncid_int)
    if ncid_int<=22:
        chr=f"chr{ncid_int}"
    elif ncid_int==23:
        chr="chrX"
    elif ncid_int==24:
        chr="chrY"
    else:
        chr="chrUnk"
    return chr

In [ ]:
valid_chr = ["chr"+str(x) for x in range(1,24)]+["chrX","chrY"]

In [ ]:
refseq["chr"] = refseq["chr"].apply(ncid_to_chr)
refseq_filtered = refseq[refseq["chr"].isin(valid_chr)]
print(refseq_filtered)

In [ ]:
ensembl["chr"] = "chr"+ensembl["chr"]
ensembl_filtered = ensembl[ensembl["chr"].isin(valid_chr)]
print(ensembl_filtered)

In [ ]:
def attribute_to_dict(attribute):
    attribute = attribute+" "
    attribute_dict = {}
    for attr in attribute.split('"; ')[:-1]:
        try:
            key, value = attr.split('"')[:2]
        except:
            print(attribute)
            print(attr)
            raise ValueError
        key = key.strip()
        attribute_dict[key] = value
    return attribute_dict

In [45]:
print(ensembl_filtered)

           chr          source      feature    start      end score strand  \
0         chr1  ensembl_havana         gene  3069168  3438621     .      +   
1         chr1          havana   transcript  3069168  3434342     .      +   
2         chr1          havana         exon  3069168  3069296     .      +   
3         chr1          havana          CDS  3069260  3069296     .      +   
4         chr1          havana  start_codon  3069260  3069262     .      +   
...        ...             ...          ...      ...      ...   ...    ...   
4102141  chr21   havana_tagene         exon  9678230  9678353     .      +   
4102142  chr21   havana_tagene         exon  9679515  9679891     .      +   
4102143  chr21   havana_tagene   transcript  9662173  9677705     .      +   
4102144  chr21   havana_tagene         exon  9662173  9662338     .      +   
4102145  chr21   havana_tagene         exon  9677372  9677705     .      +   

        frame                                          attribut

In [ ]:
print(ensembl_filtered["feature"].value_counts())
print(refseq_filtered["feature"].value_counts())

In [38]:
print(refseq_filtered[refseq_filtered["feature"]=="exon"]["attribute"].values[0])

gene_id "DDX11L1"; transcript_id "NR_046018.2"; db_xref "GeneID:100287102"; db_xref "GenBank:NR_046018.2"; db_xref "HGNC:HGNC:37102"; gene "DDX11L1"; product "DEAD/H-box helicase 11 like 1 (pseudogene)"; pseudo "true"; transcript_biotype "transcript"; exon_number "1"; 


In [37]:
print(ensembl_filtered[ensembl_filtered["feature"]=="exon"]["attribute"].values[0])

gene_id "ENSG00000142611"; gene_version "17"; transcript_id "ENST00000511072"; transcript_version "5"; exon_number "1"; gene_name "PRDM16"; gene_source "ensembl_havana"; gene_biotype "protein_coding"; transcript_name "PRDM16-206"; transcript_source "havana"; transcript_biotype "protein_coding"; exon_id "ENSE00002048533"; exon_version "1"; tag "gencode_basic"; tag "gencode_primary"; transcript_support_level "5";


In [ ]:
ensembl_filtered = ensembl_filtered.copy().reset_index(drop=True)
ensembl_filtered["attribute_dict"] = ensembl_filtered["attribute"].apply(attribute_to_dict)

In [ ]:
refseq_filtered = refseq_filtered.copy().reset_index(drop=True)
refseq_filtered["attribute_dict"] = refseq_filtered["attribute"].apply(attribute_to_dict)

In [ ]:
ensembl_filtered["gene_name"] = ensembl_filtered["attribute_dict"].apply(lambda x: x.get("gene_name", "NA"))
refseq_filtered["gene_name"] = refseq_filtered["attribute_dict"].apply(lambda x: x.get("gene_id", "NA"))

In [ ]:
ensembl_filtered["transcript_id"] = ensembl_filtered["attribute_dict"].apply(lambda x: x.get("transcript_id", "NA"))
refseq_filtered["transcript_id"] = refseq_filtered["attribute_dict"].apply(lambda x: x.get("transcript_id", "NA"))

In [40]:
ensembl_filtered["gene_id"] = ensembl_filtered["attribute_dict"].apply(lambda x: x.get("gene_id", "NA"))
refseq_filtered["gene_id"] = refseq_filtered["attribute_dict"].apply(lambda x: x.get("gene_id", "NA"))

In [42]:
ensembl_filtered["gene_name"] = ensembl_filtered.apply(lambda x: x["gene_id"] if x["gene_name"]=="NA" else x["gene_name"], axis=1)

In [44]:
print(ensembl_filtered[ensembl_filtered["gene_name"]=="MALAT1"])
print(refseq_filtered[refseq_filtered["gene_name"]=="MALAT1"])

           chr         source     feature     start       end score strand  \
2407440  chr11         havana        gene  65497606  65508073     .      +   
2407441  chr11  havana_tagene  transcript  65497606  65508073     .      +   
2407442  chr11  havana_tagene        exon  65497606  65508073     .      +   
2407443  chr11         havana  transcript  65497688  65499847     .      +   
2407444  chr11         havana        exon  65497688  65497865     .      +   
...        ...            ...         ...       ...       ...   ...    ...   
2407710  chr11  havana_tagene        exon  65498969  65499966     .      +   
2407711  chr11  havana_tagene  transcript  65503727  65504463     .      +   
2407712  chr11  havana_tagene        exon  65503727  65503931     .      +   
2407713  chr11  havana_tagene        exon  65504133  65504206     .      +   
2407714  chr11  havana_tagene        exon  65504326  65504463     .      +   

        frame                                          attribut

In [ ]:
ensembl_filtered["coding"] = ensembl_filtered["attribute_dict"].apply(lambda x: "protein_coding" in x.get("gene_biotype", "NA"))

In [ ]:
refseq_filtered = refseq_filtered[~(refseq_filtered["transcript_id"].str.startswith("unassigned"))].copy()
print(refseq_filtered)

In [ ]:
refseq_filtered["coding"] = refseq_filtered["transcript_id"].str[1] == "M"
print(refseq_filtered)

In [130]:
print(ensembl_filtered["coding"].value_counts())
print(refseq_filtered["coding"].value_counts())

coding
True     3083737
False    1018409
Name: count, dtype: int64
coding
True     3909153
False     371394
Name: count, dtype: int64


In [ ]:
refseq_filtered[["start", "end"]] = refseq_filtered[["start", "end"]].astype(int)

In [ ]:
def make_refflat(df):
    df = df[df["feature"].isin(["exon", "CDS"])]
    df = df.groupby("transcript_id")
    refflat_dict = {"transcript_id":[], "gene_id":[], "gene_name":[], "chr":[], "strand":[], "txStart":[], "txEnd":[], "cdsStart":[], "cdsEnd":[], "exonCount":[], "exonStarts":[], "exonEnds":[], "coding": []}
    for transcript_id, group in tqdm.tqdm(df, total=df.ngroups):
        refflat_dict["transcript_id"].append(transcript_id)
        refflat_dict["gene_id"].append(group["gene_id"].values[0])
        refflat_dict["gene_name"].append(group["gene_name"].values[0])
        refflat_dict["chr"].append(group["chr"].values[0])
        refflat_dict["strand"].append(group["strand"].values[0])
        start = group["start"].min()
        end = group["end"].max()
        refflat_dict["txStart"].append(start)
        refflat_dict["txEnd"].append(end)
        if group[group["feature"]=="CDS"].shape[0]==0:
            refflat_dict["cdsStart"].append(start)
            refflat_dict["cdsEnd"].append(end)
        else:
            refflat_dict["cdsStart"].append(group[group["feature"]=="CDS"]["start"].min())
            refflat_dict["cdsEnd"].append(group[group["feature"]=="CDS"]["end"].max())
        refflat_dict["coding"].append(group["coding"].any())
        refflat_dict["exonCount"].append(group[group["feature"]=="exon"].shape[0])
        refflat_dict["exonStarts"].append(",".join(group[group["feature"]=="exon"]["start"].sort_values().astype(str).values))
        refflat_dict["exonEnds"].append(",".join(group[group["feature"]=="exon"]["end"].sort_values().astype(str).values))
    refflat = pd.DataFrame(refflat_dict)
    return refflat



In [137]:
ensembl_refflat = make_refflat(ensembl_filtered)

100%|█| 385622/385622 [15


In [138]:
refseq_refflat = make_refflat(refseq_filtered)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 177431/177431 [07:37<00:00, 388.25it/s]


In [139]:
refseq_refflat["source"] = "refseq"
ensembl_refflat["source"] = "ensembl"

In [140]:
ensembl_refflat.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/anno/ensembl_refflat.pkl")

In [141]:
refseq_refflat.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/anno/refseq_refflat.pkl")

In [142]:
print(ensembl_refflat)

          transcript_id          gene_id  gene_name    chr strand    txStart  \
0       ENST00000000233  ENSG00000004059       ARF5   chr7      +  127588411   
1       ENST00000000412  ENSG00000003056       M6PR  chr12      -    8940361   
2       ENST00000000442  ENSG00000173153      ESRRA  chr11      +   64305524   
3       ENST00000001008  ENSG00000004478      FKBP4  chr12      +    2794970   
4       ENST00000001146  ENSG00000003137    CYP26B1   chr2      -   72129238   
...                 ...              ...        ...    ...    ...        ...   
385617  ENST00000850839  ENSG00000310540  LINC02968   chrY      +    1661795   
385618  ENST00000850840  ENSG00000292360  LINC03112   chrY      -    2566027   
385619  ENST00000850841  ENSG00000292360  LINC03112   chrY      -    2566027   
385620  ENST00000850842  ENSG00000292360  LINC03112   chrY      -    2566024   
385621  ENST00000850843  ENSG00000292362     CD99P1   chrY      +    2609381   

            txEnd   cdsStart     cdsEnd

In [143]:
print(refseq_refflat)

       transcript_id       gene_id     gene_name    chr strand    txStart  \
0        NM_000014.6           A2M           A2M  chr12      -    9067708   
1        NM_000015.3          NAT2          NAT2   chr8      +   18391282   
2        NM_000016.6         ACADM         ACADM   chr1      +   75724709   
3        NM_000017.4         ACADS         ACADS  chr12      +  120725826   
4        NM_000018.4        ACADVL        ACADVL  chr17      +    7219938   
...              ...           ...           ...    ...    ...        ...   
177426   XR_951226.3  LOC105372853  LOC105372853  chr22      +   18057028   
177427   XR_951228.1  LOC105379518  LOC105379518  chr22      +   18233149   
177428   XR_951230.2       FAM247D       FAM247D  chr22      +   18349976   
177429   XR_951236.3      TMEM191B      TMEM191B  chr22      +   18527802   
177430   XR_951292.2     LINC02968     LINC02968   chrX      +    1732556   

            txEnd   cdsStart     cdsEnd  exonCount  \
0         9115919    

In [144]:
ensembl_refflat_grouped = ensembl_refflat.groupby(["chr", "strand"])
ensembl_refflat_grouped = {key: value for key, value in ensembl_refflat_grouped}
refseq_refflat_grouped = refseq_refflat.groupby(["chr", "strand"])
refseq_refflat_grouped = {key: value for key, value in refseq_refflat_grouped}

In [145]:
assert (ensembl_refflat_grouped.keys() == refseq_refflat_grouped.keys())

In [146]:
merged_refflat = []

for key in tqdm.tqdm(ensembl_refflat_grouped.keys()):
    ensembl = ensembl_refflat_grouped[key]
    refseq = refseq_refflat_grouped[key]
    ## Merge and remove duplicates
    merged = pd.concat([ensembl, refseq], axis=0)
    merged = merged.drop_duplicates(subset=["exonStarts", "exonEnds"], keep="first")
    merged_refflat.append(merged)

merged_refflat = pd.concat(merged_refflat, axis=0)

print(merged_refflat)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48/48 [00:01<00:00, 47.36it/s]


          transcript_id          gene_id     gene_name   chr strand  \
14      ENST00000003912  ENSG00000001461        NIPAL3  chr1      +   
53      ENST00000008440  ENSG00000010072         SPRTN  chr1      +   
58      ENST00000009105  ENSG00000008118        CAMK1G  chr1      +   
63      ENST00000010299  ENSG00000009780        FAM76A  chr1      +   
71      ENST00000011700  ENSG00000048707        VPS13D  chr1      +   
...                 ...              ...           ...   ...    ...   
174741      XR_938669.2     LOC105377241  LOC105377241  chrY      -   
174742      XR_938670.2     LOC105377241  LOC105377241  chrY      -   
174745      XR_938673.3          REREP2Y       REREP2Y  chrY      -   
174746      XR_938675.3     LOC105377244  LOC105377244  chrY      -   
177350      XR_950583.3      LINC03112_1   LINC03112_1  chrY      -   

          txStart      txEnd   cdsStart     cdsEnd  exonCount  \
14       24415803   24475252   24442139   24469182         13   
53      231338256

In [147]:
merged_refflat.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/anno/merged_refflat.pkl")

In [148]:
print(len(ensembl_refflat), len(refseq_refflat), len(merged_refflat))

385622 177431 537999


In [149]:
print(merged_refflat[merged_refflat["source"]=="ensembl"])
print(merged_refflat[merged_refflat["source"]=="refseq"])

          transcript_id          gene_id  gene_name   chr strand    txStart  \
14      ENST00000003912  ENSG00000001461     NIPAL3  chr1      +   24415803   
53      ENST00000008440  ENSG00000010072      SPRTN  chr1      +  231338256   
58      ENST00000009105  ENSG00000008118     CAMK1G  chr1      +  209583717   
63      ENST00000010299  ENSG00000009780     FAM76A  chr1      +   27726057   
71      ENST00000011700  ENSG00000048707     VPS13D  chr1      +   12277121   
...                 ...              ...        ...   ...    ...        ...   
385615  ENST00000850837  ENSG00000292360  LINC03112  chrY      -    2566025   
385616  ENST00000850838  ENSG00000292360  LINC03112  chrY      -    2566027   
385618  ENST00000850840  ENSG00000292360  LINC03112  chrY      -    2566027   
385619  ENST00000850841  ENSG00000292360  LINC03112  chrY      -    2566027   
385620  ENST00000850842  ENSG00000292360  LINC03112  chrY      -    2566024   

            txEnd   cdsStart     cdsEnd  exonCount 

In [150]:
merged_refflat_groupby_gene = merged_refflat.groupby("gene_name")
merged_refflat_groupby_gene = {key: value for key, value in merged_refflat_groupby_gene}

gene_id
1      70425
2      20101
3        326
4          7
7          4
8          3
6          2
9          1
5          1
170        1
27         1
19         1
50         1
33         1
22         1
756        1
Name: count, dtype: int64


In [159]:
passed = []
failed = []

for gene_name, group in tqdm.tqdm(merged_refflat_groupby_gene):
    unique_id = group["gene_id"].unique()
    if group["source"].nunique()==1:
        passed.append(group)
    else:
        if len(unique_id)==1:
            passed.append(group)
        elif len(unique_id)==2:
            group = group.copy()
            ensembl_id = group[group["source"]=="ensembl"]["gene_id"].values[0]
            group["gene_id"] = ensembl_id
            passed.append(group)
        else:
            failed.append(group)

print(len(passed), len(failed))
                

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 90877/90877 [00:43<00:00, 2099.28it/s]

90548 329


In [160]:
merged_refflat_v2 = pd.concat(passed+failed, axis=0)
print(merged_refflat_v2)

          transcript_id          gene_id gene_name    chr strand    txStart  \
200050  ENST00000648701  ENSG00000285609   5S_rRNA   chr1      -  182944365   
223894  ENST00000674448  ENSG00000288601   5S_rRNA  chr16      -   47450347   
183361  ENST00000618379  ENSG00000277488   5S_rRNA  chr17      -   37940704   
200151  ENST00000648813  ENSG00000285674   5S_rRNA  chr19      -    7886977   
180164  ENST00000612131  ENSG00000276861   5S_rRNA   chr2      +   89600571   
...                 ...              ...       ...    ...    ...        ...   
357059  ENST00000821490  ENSG00000293494    ZPLD2P   chr1      +   26225438   
357060  ENST00000821491  ENSG00000293494    ZPLD2P   chr1      +   26225087   
357061  ENST00000821492  ENSG00000293494    ZPLD2P   chr1      +   26225315   
357062  ENST00000821493  ENSG00000293494    ZPLD2P   chr1      +   26225320   
76242       NR_110698.1           ZPLD2P    ZPLD2P   chr1      +   26225320   

            txEnd   cdsStart     cdsEnd  exonCount 

In [161]:
merged_refflat_v2 = merged_refflat_v2.reset_index(drop=True)
merged_refflat_v2.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/anno/merged_refflat_v2.pkl")

In [162]:
refflat_columns = ["gene_name", "transcript_id", "chr", "strand", "txStart", "txEnd", "cdsStart", "cdsEnd", "exonCount", "exonStarts", "exonEnds"]
refflat_for_record = merged_refflat_v2[refflat_columns].copy()
refflat_for_record.to_csv("/extdata4/baeklab/Hyeonseo/m6A/anno/refflat_for_record.txt", sep="\t", index=False, header=False)

In [163]:
print(refseq_filtered)

          chr      source     feature     start       end score strand frame  \
0        chr1  BestRefSeq        gene     11874     14409     .      +     .   
1        chr1  BestRefSeq  transcript     11874     14409     .      +     .   
2        chr1  BestRefSeq        exon     11874     12227     .      +     .   
3        chr1  BestRefSeq        exon     12613     12721     .      +     .   
4        chr1  BestRefSeq        exon     13221     14409     .      +     .   
...       ...         ...         ...       ...       ...   ...    ...   ...   
4290478  chrY  BestRefSeq        gene  57212178  57214703     .      -     .   
4290479  chrY  BestRefSeq  transcript  57212178  57214703     .      -     .   
4290480  chrY  BestRefSeq        exon  57214350  57214703     .      -     .   
4290481  chrY  BestRefSeq        exon  57213856  57213964     .      -     .   
4290482  chrY  BestRefSeq        exon  57212178  57213357     .      -     .   

                                       

In [164]:
print(ensembl_filtered)

           chr          source      feature    start      end score strand  \
0         chr1  ensembl_havana         gene  3069168  3438621     .      +   
1         chr1          havana   transcript  3069168  3434342     .      +   
2         chr1          havana         exon  3069168  3069296     .      +   
3         chr1          havana          CDS  3069260  3069296     .      +   
4         chr1          havana  start_codon  3069260  3069262     .      +   
...        ...             ...          ...      ...      ...   ...    ...   
4102141  chr21   havana_tagene         exon  9678230  9678353     .      +   
4102142  chr21   havana_tagene         exon  9679515  9679891     .      +   
4102143  chr21   havana_tagene   transcript  9662173  9677705     .      +   
4102144  chr21   havana_tagene         exon  9662173  9662338     .      +   
4102145  chr21   havana_tagene         exon  9677372  9677705     .      +   

        frame                                          attribut

In [166]:
ensembl_transcript_unique_refflat = merged_refflat_v2[merged_refflat_v2["source"]=="ensembl"]["transcript_id"].unique()
refseq_transcript_unique_refflat = merged_refflat_v2[merged_refflat_v2["source"]=="refseq"]["transcript_id"].unique()
print(len(ensembl_transcript_unique_refflat), len(refseq_transcript_unique_refflat))

385465 152534


In [167]:
ensembl_filtered_transcript = []
refseq_filtered_transcript = []

ensembl_filtered_grouped = ensembl_filtered.groupby("transcript_id")
refseq_filtered_grouped = refseq_filtered.groupby("transcript_id")

for transcript_id in tqdm.tqdm(ensembl_transcript_unique_refflat):
    ensembl_filtered_transcript.append(ensembl_filtered_grouped.get_group(transcript_id))

for transcript_id in tqdm.tqdm(refseq_transcript_unique_refflat):
    refseq_filtered_transcript.append(refseq_filtered_grouped.get_group(transcript_id))
    

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 152534/152534 [00:29<00:00, 5196.54it/s]


In [168]:
transcript_to_gene_dict = dict(zip(merged_refflat_v2["transcript_id"], merged_refflat_v2["gene_id"]))

In [169]:
merged_gtf = pd.concat(ensembl_filtered_transcript+refseq_filtered_transcript, axis=0)
merged_gtf["new_gene_id"] = merged_gtf["transcript_id"].apply(lambda x: transcript_to_gene_dict[x])
print(merged_gtf)

           chr      source     feature      start        end score strand  \
36503     chr1     ensembl  transcript  182944365  182944490     .      -   
36504     chr1     ensembl        exon  182944365  182944490     .      -   
3272176  chr16     ensembl  transcript   47450347   47450461     .      -   
3272177  chr16     ensembl        exon   47450347   47450461     .      -   
3420710  chr17     ensembl  transcript   37940704   37940790     .      -   
...        ...         ...         ...        ...        ...   ...    ...   
88849     chr1  BestRefSeq  transcript   26225320   26229840     .      +   
88850     chr1  BestRefSeq        exon   26225320   26225703     .      +   
88851     chr1  BestRefSeq        exon   26227404   26227555     .      +   
88852     chr1  BestRefSeq        exon   26227794   26227996     .      +   
88853     chr1  BestRefSeq        exon   26229502   26229840     .      +   

        frame                                          attribute  \
36503  

In [170]:
merged_gtf = merged_gtf.reset_index(drop=True)
print(merged_gtf)

           chr      source     feature      start        end score strand  \
0         chr1     ensembl  transcript  182944365  182944490     .      -   
1         chr1     ensembl        exon  182944365  182944490     .      -   
2        chr16     ensembl  transcript   47450347   47450461     .      -   
3        chr16     ensembl        exon   47450347   47450461     .      -   
4        chr17     ensembl  transcript   37940704   37940790     .      -   
...        ...         ...         ...        ...        ...   ...    ...   
7735344   chr1  BestRefSeq  transcript   26225320   26229840     .      +   
7735345   chr1  BestRefSeq        exon   26225320   26225703     .      +   
7735346   chr1  BestRefSeq        exon   26227404   26227555     .      +   
7735347   chr1  BestRefSeq        exon   26227794   26227996     .      +   
7735348   chr1  BestRefSeq        exon   26229502   26229840     .      +   

        frame                                          attribute  \
0      

In [174]:
## update attribute_dict to have new gene_id

def update_attribute_dict(attribute_dict, gene_id):
    attribute_dict["gene_id"] = gene_id
    return attribute_dict

In [182]:
merged_gtf["attribute_dict"] = merged_gtf.apply(lambda x: update_attribute_dict(x["attribute_dict"], x["new_gene_id"]), axis=1)

In [185]:
gene_gtf_df = pd.concat([refseq_filtered[refseq_filtered["feature"]=="gene"], ensembl_filtered[ensembl_filtered["feature"]=="gene"]], axis=0)
print(gene_gtf_df)


           chr      source feature     start       end score strand frame  \
0         chr1  BestRefSeq    gene     11874     14409     .      +     .   
5         chr1  BestRefSeq    gene     14362     29370     .      -     .   
18        chr1  BestRefSeq    gene     17369     17436     .      -     .   
25        chr1      Gnomon    gene     29774     35418     .      +     .   
30        chr1  BestRefSeq    gene     30366     30503     .      +     .   
...        ...         ...     ...       ...       ...   ...    ...   ...   
4101459  chr21      havana    gene  34412200  34412587     .      +     .   
4101462  chr21      havana    gene  34414494  34423951     .      -     .   
4101483  chr21      havana    gene  34425508  34426017     .      +     .   
4101486  chr21      havana    gene   6630182   6670761     .      -     .   
4102077  chr21      havana    gene   9529928   9679906     .      +     .   

                                                 attribute  \
0        gene

In [199]:
gene_gtf_df

,chr,source,feature,start,end,score,strand,frame,attribute,attribute_dict,gene_name,transcript_id,gene_id,coding
0,chr1,BestRefSeq,gene,11874,14409,.,+,.,"gene_id ""DDX11L1""; transcript_id """"; db_xref ""...","{'gene_id': 'DDX11L1', 'transcript_id': '', 'd...",DDX11L1,,DDX11L1,False
1,chr1,BestRefSeq,gene,14362,29370,.,-,.,"gene_id ""WASH7P""; transcript_id """"; db_xref ""G...","{'gene_id': 'WASH7P', 'transcript_id': '', 'db...",WASH7P,,WASH7P,False
2,chr1,BestRefSeq,gene,17369,17436,.,-,.,"gene_id ""MIR6859-1""; transcript_id """"; db_xref...","{'gene_id': 'MIR6859-1', 'transcript_id': '', ...",MIR6859-1,,MIR6859-1,False
3,chr1,Gnomon,gene,29774,35418,.,+,.,"gene_id ""MIR1302-2HG""; transcript_id """"; db_xr...","{'gene_id': 'MIR1302-2HG', 'transcript_id': ''...",MIR1302-2HG,,MIR1302-2HG,False
4,chr1,BestRefSeq,gene,30366,30503,.,+,.,"gene_id ""MIR1302-2""; transcript_id """"; db_xref...","{'gene_id': 'MIR1302-2', 'transcript_id': '', ...",MIR1302-2,,MIR1302-2,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137612,chr21,havana,gene,34412200,34412587,.,+,.,"gene_id ""ENSG00000273104""; gene_version ""1""; g...","{'gene_id': 'ENSG00000273104', 'gene_version':...",ENSG00000273104,NA,ENSG00000273104,False
137613,chr21,havana,gene,34414494,34423951,.,-,.,"gene_id ""ENSG00000243627""; gene_version ""6""; g...","{'gene_id': 'ENSG00000243627', 'gene_version':...",SMIM34,NA,ENSG00000243627,True
137614,chr21,havana,gene,34425508,34426017,.,+,.,"gene_id ""ENSG00000272958""; gene_version ""1""; g...","{'gene_id': 'ENSG00000272958', 'gene_version':...",ENSG00000272958,NA,ENSG00000272958,False
137615,chr21,havana,gene,6630182,6670761,.,-,.,"gene_id ""ENSG00000280145""; gene_version ""4""; g...","{'gene_id': 'ENSG00000280145', 'gene_version':...",ENSG00000280145,NA,ENSG00000280145,False


In [187]:
gene_gtf_df = gene_gtf_df.reset_index(drop=True)
assert(gene_gtf_df["gene_id"].nunique()==gene_gtf_df.shape[0])

In [188]:
merged_gtf["gene_id"] = merged_gtf["new_gene_id"]
merged_gtf = merged_gtf.drop(columns=["new_gene_id"])

In [196]:
def format_attribute(attribute_dict):
    attribute = ""
    for key, value in attribute_dict.items():
        attribute+=f'{key} "{value}"; '
    attribute = attribute[:-1]
    return attribute

In [197]:
merged_gtf["new_attribute"] = merged_gtf["attribute_dict"].apply(format_attribute)
print(merged_gtf)

           chr      source     feature      start        end score strand  \
0         chr1     ensembl  transcript  182944365  182944490     .      -   
1         chr1     ensembl        exon  182944365  182944490     .      -   
2        chr16     ensembl  transcript   47450347   47450461     .      -   
3        chr16     ensembl        exon   47450347   47450461     .      -   
4        chr17     ensembl  transcript   37940704   37940790     .      -   
...        ...         ...         ...        ...        ...   ...    ...   
7735344   chr1  BestRefSeq  transcript   26225320   26229840     .      +   
7735345   chr1  BestRefSeq        exon   26225320   26225703     .      +   
7735346   chr1  BestRefSeq        exon   26227404   26227555     .      +   
7735347   chr1  BestRefSeq        exon   26227794   26227996     .      +   
7735348   chr1  BestRefSeq        exon   26229502   26229840     .      +   

        frame                                          attribute  \
0      

In [198]:
merged_gtf["attribute"] = merged_gtf["new_attribute"]
merged_gtf.drop(columns=["attribute_dict", "new_attribute"], inplace=True)
print(merged_gtf)

           chr      source     feature      start        end score strand  \
0         chr1     ensembl  transcript  182944365  182944490     .      -   
1         chr1     ensembl        exon  182944365  182944490     .      -   
2        chr16     ensembl  transcript   47450347   47450461     .      -   
3        chr16     ensembl        exon   47450347   47450461     .      -   
4        chr17     ensembl  transcript   37940704   37940790     .      -   
...        ...         ...         ...        ...        ...   ...    ...   
7735344   chr1  BestRefSeq  transcript   26225320   26229840     .      +   
7735345   chr1  BestRefSeq        exon   26225320   26225703     .      +   
7735346   chr1  BestRefSeq        exon   26227404   26227555     .      +   
7735347   chr1  BestRefSeq        exon   26227794   26227996     .      +   
7735348   chr1  BestRefSeq        exon   26229502   26229840     .      +   

        frame                                          attribute gene_name 

In [200]:
gene_gtf_df.drop(columns=["attribute_dict"], inplace=True)
print(gene_gtf_df)

          chr      source feature     start       end score strand frame  \
0        chr1  BestRefSeq    gene     11874     14409     .      +     .   
1        chr1  BestRefSeq    gene     14362     29370     .      -     .   
2        chr1  BestRefSeq    gene     17369     17436     .      -     .   
3        chr1      Gnomon    gene     29774     35418     .      +     .   
4        chr1  BestRefSeq    gene     30366     30503     .      +     .   
...       ...         ...     ...       ...       ...   ...    ...   ...   
137612  chr21      havana    gene  34412200  34412587     .      +     .   
137613  chr21      havana    gene  34414494  34423951     .      -     .   
137614  chr21      havana    gene  34425508  34426017     .      +     .   
137615  chr21      havana    gene   6630182   6670761     .      -     .   
137616  chr21      havana    gene   9529928   9679906     .      +     .   

                                                attribute        gene_name  \
0       g

In [202]:
merged_gtf =  merged_gtf.reset_index(drop=True)

In [203]:
merged_gtf_grouped = merged_gtf.groupby("gene_id")

In [204]:
chr_to_int_dict = {f"chr{x}": x for x in range(1,23)}
chr_to_int_dict["chrX"] = 23
chr_to_int_dict["chrY"] = 24

In [ ]:
gtf_dict_for_sorting = {}

for gene_id, group in tqdm.tqdm(merged_gtf_grouped):
    gene_row = gene_gtf_df[gene_gtf_df["gene_id"]==gene_id].copy()
    if gene_row.shape[0]==0:
        continue
    start = group["start"].min()
    end = group["end"].max()
    gene_row["start"] = start
    gene_row["end"] = end
    chr = chr_to_int_dict[gene_row["chr"].values[0]]
    key = (chr, start, end, gene_id)
    group = group.groupby("transcript_id")
    sort_dict = {key: (value["start"].values[0], value["end"].values[0], value["transcript_id"].values[0]) for key, value in group}
    sorted_key = sorted(sort_dict.keys(), key=lambda x: sort_dict[x])
    gene_df = pd.concat([gene_row]+[group.get_group(key) for key in sorted_key], axis=0)
    gtf_dict_for_sorting[key] = gene_df

gtf_dict_for_sorting = [gtf_dict_for_sorting[key] for key in sorted(gtf_dict_for_sorting.keys())]
gtf_df = pd.concat(gtf_dict_for_sorting, axis=0)
    

  6%|████████▉                                                                                                                                                 | 5352/92816 [02:53<45:18, 32.17it/s]

In [194]:
print(gtf_df)

           chr          source     feature    start      end score strand  \
88509     chr7  ensembl_havana        gene  5526409  5563902     .      -   
52325     chr7          havana  transcript  5526409  5530601     .      -   
52326     chr7          havana        exon  5530542  5530601     .      -   
52327     chr7          havana        exon  5529535  5529684     .      -   
52328     chr7          havana         CDS  5529535  5529657     .      -   
...        ...             ...         ...      ...      ...   ...    ...   
3642369  chr17          havana        exon  7675994  7676272     .      -   
3642370  chr17          havana        exon  7675053  7675236     .      -   
3642371  chr17          havana        exon  7674526  7674971     .      -   
3642466  chr17          havana  transcript  7685249  7687523     .      -   
3642467  chr17          havana        exon  7685249  7687523     .      -   

        frame                                          attribute  \
88509  

In [ ]:
gtf_df.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/anno/merged_gtf.pkl")


In [ ]:
columns = ["chr", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]
gtf_df_export = gtf_df[columns].copy()
gtf_df_export.to_csv("/extdata4/baeklab/Hyeonseo/m6A/anno/merged_gtf_manual.gtf", sep="\t", index=False, header=False)